<a href="https://colab.research.google.com/github/Briyad37/Convolutional-Neural-Networks-TomatoDoc-Models/blob/main/Remove_duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Checking for duplicates in training and testing datasets.

In [19]:
import os
import numpy as np
import pandas as pd
import cv2
import shutil

In [20]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
dataset_directory_location = '/content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset'

In [22]:
print(os.getcwd())

/content


In [23]:
os.listdir(dataset_directory_location)

['test', 'train']

In [24]:
training_part = os.path.join(dataset_directory_location, 'train')
testing_part = os.path.join(dataset_directory_location, 'test')

Cleaning part

# 2. Configuration

In [25]:
FOLDER1 = training_part
FOLDER2 = testing_part

REMOVED_FOLDER = "removed_duplicates"

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

DUPLICATE_THRESHOLD = 5

In [26]:
CLEAN_DATASET_ROOT = os.path.join(dataset_directory_location, 'clean_dataset_no_duplicates')


 3. Load and prepare image


In [27]:


def load_and_prepare_image(image_path):
    image = cv2.imread(image_path)

    if image is None:
        return None

    image = cv2.resize(image, (300, 300))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    return image


 4. Compare two images


In [28]:


def compare_images(image1, image2):
    difference = cv2.absdiff(image1, image2)
    score = np.mean(difference)

    return score


 5. Move file safely

In [29]:


def move_file_safely(file_path, destination_folder):
    filename = os.path.basename(file_path)
    destination_path = os.path.join(destination_folder, filename)

    name, extension = os.path.splitext(filename)
    counter = 1

    while os.path.exists(destination_path):
        new_filename = f"{name}_{counter}{extension}"
        destination_path = os.path.join(destination_folder, new_filename)
        counter += 1

    shutil.move(file_path, destination_path)

6. Check if file is image

In [30]:
def is_image_file(filename):
    return filename.lower().endswith(IMAGE_EXTENSIONS)

# 7. Find duplicate images


In [31]:
def find_duplicates():
    duplicates_found = []

    # Collect all image files from FOLDER1 recursively
    images_in_folder1 = []
    for root, _, files in os.walk(FOLDER1):
        for file in files:
            if is_image_file(file):
                images_in_folder1.append(os.path.join(root, file))

    # Collect all image files from FOLDER2 recursively
    images_in_folder2 = []
    for root, _, files in os.walk(FOLDER2):
        for file in files:
            if is_image_file(file):
                images_in_folder2.append(os.path.join(root, file))

    # Compare images between FOLDER1 and FOLDER2
    for path1 in images_in_folder1:
        img1 = load_and_prepare_image(path1)

        if img1 is None:
            print(f"Warning: Could not load or prepare image: {path1}. Skipping.")
            continue

        for path2 in images_in_folder2:
            img2 = load_and_prepare_image(path2)

            if img2 is None:
                print(f"Warning: Could not load or prepare image: {path2}. Skipping.")
                continue

            score = compare_images(img1, img2)

            if score < DUPLICATE_THRESHOLD:
                print(f"Duplicate found: {os.path.basename(path1)} <--> {os.path.basename(path2)}")
                print(f"Difference score: {score}")

                duplicates_found.append((path1, path2))

    return duplicates_found

 8. Remove duplicate images

In [32]:
def remove_duplicates(duplicates_found):
    os.makedirs(REMOVED_FOLDER, exist_ok=True)

    for path1, path2 in duplicates_found:

        if os.path.exists(path1):
            move_file_safely(path1, REMOVED_FOLDER)

        if os.path.exists(path2):
            move_file_safely(path2, REMOVED_FOLDER)

# 9. Main cleaning function

In [33]:
def copy_images_with_structure(source_folder, target_base_folder, paths_to_skip=None):
    if paths_to_skip is None:
        paths_to_skip = set()

    print(f"Copying unique images from {source_folder} to {target_base_folder}...")

    for root, _, files in os.walk(source_folder):
        relative_path_from_source = os.path.relpath(root, source_folder)
        target_dir = os.path.join(target_base_folder, relative_path_from_source)
        os.makedirs(target_dir, exist_ok=True)

        for file in files:
            original_file_path = os.path.join(root, file)
            if is_image_file(file) and original_file_path not in paths_to_skip:
                shutil.copy2(original_file_path, os.path.join(target_dir, file))
    print(f"Finished copying from {source_folder}.")

def clean_duplicates(clean_dataset_root):
    print("Starting to create a new dataset with duplicates removed...")

    os.makedirs(clean_dataset_root, exist_ok=True)

    clean_folder1_path = os.path.join(clean_dataset_root, os.path.basename(FOLDER1))
    clean_folder2_path = os.path.join(clean_dataset_root, os.path.basename(FOLDER2))

    # 1. Find duplicates between FOLDER1 and FOLDER2
    duplicates_found = find_duplicates()

    if len(duplicates_found) == 0:
        print("No cross-folder duplicates found. Copying all images.")
        # If no duplicates, just copy everything
        copy_images_with_structure(FOLDER1, clean_folder1_path)
        copy_images_with_structure(FOLDER2, clean_folder2_path)

    else:
        # Collect paths from FOLDER2 that are duplicates of FOLDER1 images
        paths_to_skip_from_folder2 = set()
        for _, path2 in duplicates_found:
            paths_to_skip_from_folder2.add(path2)

        # 2. Copy all images from FOLDER1 to its clean counterpart
        copy_images_with_structure(FOLDER1, clean_folder1_path)

        # 3. Copy images from FOLDER2 to its clean counterpart, skipping duplicates of FOLDER1 images
        copy_images_with_structure(FOLDER2, clean_folder2_path, paths_to_skip_from_folder2)

        print(f"Successfully created a new clean dataset at: {clean_dataset_root}")
        print(f"Removed {len(paths_to_skip_from_folder2)} duplicate files from {os.path.basename(FOLDER2)}'s copy.")

# 10. Run the program


In [34]:
if __name__ == "__main__":
    clean_duplicates(CLEAN_DATASET_ROOT)

Starting to create a new dataset with duplicates removed...
Duplicate found: 1af0bfe1-4bcf-4b8b-be66-5d0953eb647e___GH_HL Leaf 482.2.JPG <--> cfd491d6-4af5-4728-8f0e-0d330a07174a___GH_HL Leaf 482.2.JPG
Difference score: 0.0
Duplicate found: 37aad83b-7ff8-4b35-b3ed-fb8e0f54910b___GH_HL Leaf 342.1.JPG <--> e786ac89-29fe-47e3-b49e-b9a9ee7edd9d___GH_HL Leaf 342.1.JPG
Difference score: 0.0
Copying unique images from /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/train to /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/clean_dataset_no_duplicates/train...
Finished copying from /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/train.
Copying unique images from /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/test to /content/drive/MyDrive/ColabNotebooks/Tomato_disease_detection/image_dataset/clean_dataset_no_duplicates/test...
Finished copying from /content/drive/MyDrive/ColabNote